# Philippine Staple Food Prices — Chapter 9 ML Pipeline
## Association Rule Mining: Co-Occurrence & Price Spike Patterns

**Algorithms:** Apriori vs FP-Growth  
**Transactions:** Market-Day Co-Occurrence baskets + Regional Price-Spike baskets  

---

## 0. Setup & Data Loading

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, time
warnings.filterwarnings('ignore')

# ── mlxtend (association rule mining) ───────────────────────────────────────
try:
    from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
    from mlxtend.preprocessing import TransactionEncoder
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'mlxtend', '-q'])
    from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
    from mlxtend.preprocessing import TransactionEncoder

try:
    import networkx as nx
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'networkx', '-q'])
    import networkx as nx

# ── Load master dataset ──────────────────────────────────────────────────────
df = pd.read_csv('final_master_dataset_fixed.csv', parse_dates=['date'])
df = df.sort_values(['adm1_name', 'cm_name', 'date']).reset_index(drop=True)

# Strip market-type suffix for cleaner item labels
# e.g.  'Rice (regular, milled) - Retail'  →  'Rice (regular, milled)'
df['item'] = df['cm_name'].str.rsplit(' - ', n=1).str[0].str.strip()

print(f'Shape       : {df.shape}')
print(f'Date range  : {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Regions     : {df["adm1_name"].nunique()}')
print(f'Markets     : {df["mkt_name"].nunique()}')
print(f'Unique items: {df["item"].nunique()} (after suffix removal)')
df.head()

## 1. Transaction Engineering

### 1-A  Co-Occurrence Transactions (Market × Date Baskets)

**Idea:** For each *(market, date)* pair, collect the set of commodities traded.
A rule like `{Rice} → {Vegetables}` means markets that sell rice also tend to sell vegetables on the same day.

In [ ]:
# Use 2018-2021 window to keep basket count manageable
recent = df[df['year'] >= 2018].copy()

co_baskets = (
    recent
    .groupby(['mkt_name', 'date'])['item']
    .apply(lambda x: list(set(x)))   # unique items per basket
    .reset_index(name='items')
)

# Keep only baskets with ≥ 2 items (rules need at least 2)
co_baskets = co_baskets[co_baskets['items'].apply(len) >= 2].reset_index(drop=True)
co_transactions = co_baskets['items'].tolist()

print('=== Co-Occurrence Transactions ===')
print(f'Total baskets      : {len(co_transactions):,}')
print(f'Avg items/basket   : {np.mean([len(t) for t in co_transactions]):.1f}')
print(f'Max items/basket   : {max(len(t) for t in co_transactions)}')
print()
print('Sample baskets:')
for t in co_transactions[:3]:
    print(f'  {sorted(t)[:6]} ...' if len(t) > 6 else f'  {sorted(t)}')

### 1-B  Price Spike Transactions (Region × Date Spike Baskets)

**Idea:** A commodity is *spiking* on a given day if its price is at or above its commodity-specific 75th percentile.
For each *(region, date)* pair, collect the set of spiking items.
A rule like `{Fish} → {Vegetables}` means when fish prices spike, vegetable prices tend to spike too.

In [ ]:
# Per-commodity 75th percentile (normalises across different price scales)
p75 = (
    df.groupby('item')['mp_price']
    .quantile(0.75)
    .rename('p75')
    .reset_index()
)
df_sp = df.merge(p75, on='item')
df_sp['is_spike'] = df_sp['mp_price'] >= df_sp['p75']

spike_baskets = (
    df_sp[df_sp['is_spike']]
    .groupby(['adm1_name', 'date'])['item']
    .apply(lambda x: list(set(x)))
    .reset_index(name='items')
)
spike_baskets = spike_baskets[spike_baskets['items'].apply(len) >= 2].reset_index(drop=True)
spike_transactions = spike_baskets['items'].tolist()

print('=== Price Spike Transactions ===')
print(f'Spike threshold    : >= 75th percentile per commodity')
print(f'Total baskets      : {len(spike_transactions):,}')
print(f'Avg items/basket   : {np.mean([len(t) for t in spike_transactions]):.1f}')
print(f'Max items/basket   : {max(len(t) for t in spike_transactions)}')
print()
print('Sample spike baskets:')
for t in spike_transactions[:3]:
    print(f'  {sorted(t)[:6]} ...' if len(t) > 6 else f'  {sorted(t)}')

### 1-C  Transaction Encoding

Convert raw item lists to boolean DataFrames required by mlxtend.

In [ ]:
MAX_ROWS = 50_000   # cap for Apriori speed; FP-Growth uses full set

def encode(transactions, label=''):
    te = TransactionEncoder()
    arr = te.fit_transform(transactions)
    encoded = pd.DataFrame(arr, columns=te.columns_)
    sample  = encoded.sample(min(MAX_ROWS, len(encoded)), random_state=42)
    print(f'{label}:')
    print(f'  Full encoded shape   : {encoded.shape}')
    print(f'  Sample for Apriori   : {sample.shape}')
    return encoded, sample, te.columns_

co_enc,    co_samp,    co_cols    = encode(co_transactions,    'Co-Occurrence')
print()
spike_enc, spike_samp, spike_cols = encode(spike_transactions, 'Price Spike')

## 2. Association Rule Mining — Apriori

Apriori generates candidate itemsets level-by-level and prunes those below `min_support`.
It is transparent and easy to interpret, but slower on large datasets.

### 2-A  Apriori — Co-Occurrence Transactions

In [ ]:
MIN_SUP_CO    = 0.05   # itemset must appear in >= 5% of market-day baskets
MIN_SUP_SPIKE = 0.10   # itemset must appear in >= 10% of spike baskets
MIN_CONF      = 0.50   # rule confidence >= 50%

print('Running Apriori on Co-Occurrence transactions ...')
t0 = time.time()
co_freq_ap = apriori(co_samp, min_support=MIN_SUP_CO,
                     use_colnames=True, max_len=4)
apriori_co_time = time.time() - t0

co_rules_ap = association_rules(
    co_freq_ap, metric='confidence', min_threshold=MIN_CONF
)

print(f'Min support          : {MIN_SUP_CO}')
print(f'Min confidence       : {MIN_CONF}')
print(f'Frequent itemsets    : {len(co_freq_ap):,}')
print(f'Rules generated      : {len(co_rules_ap):,}')
print(f'Runtime              : {apriori_co_time:.2f}s')
print()

top10 = co_rules_ap.nlargest(10, 'lift')[['antecedents','consequents','support','confidence','lift']]
top10['antecedents'] = top10['antecedents'].apply(lambda s: ', '.join(sorted(s)))
top10['consequents'] = top10['consequents'].apply(lambda s: ', '.join(sorted(s)))
print('Top 10 Co-Occurrence Rules by Lift:')
print(top10.round(4).to_string(index=False))

### 2-B  Apriori — Price Spike Transactions

In [ ]:
print('Running Apriori on Price Spike transactions ...')
t0 = time.time()
spike_freq_ap = apriori(spike_samp, min_support=MIN_SUP_SPIKE,
                        use_colnames=True, max_len=4)
apriori_spike_time = time.time() - t0

spike_rules_ap = association_rules(
    spike_freq_ap, metric='confidence', min_threshold=MIN_CONF
)

print(f'Min support          : {MIN_SUP_SPIKE}')
print(f'Min confidence       : {MIN_CONF}')
print(f'Frequent itemsets    : {len(spike_freq_ap):,}')
print(f'Rules generated      : {len(spike_rules_ap):,}')
print(f'Runtime              : {apriori_spike_time:.2f}s')
print()

top10s = spike_rules_ap.nlargest(10, 'lift')[['antecedents','consequents','support','confidence','lift']]
top10s['antecedents'] = top10s['antecedents'].apply(lambda s: ', '.join(sorted(s)))
top10s['consequents'] = top10s['consequents'].apply(lambda s: ', '.join(sorted(s)))
print('Top 10 Price Spike Rules by Lift:')
print(top10s.round(4).to_string(index=False))

## 3. Association Rule Mining — FP-Growth

FP-Growth compresses transactions into an FP-tree and mines patterns without repeated candidate generation.
It avoids candidate explosion and is significantly faster than Apriori on large datasets.

### 3-A  FP-Growth — Co-Occurrence Transactions

In [ ]:
print('Running FP-Growth on Co-Occurrence transactions ...')
t0 = time.time()
co_freq_fp = fpgrowth(co_enc, min_support=MIN_SUP_CO,   # full dataset
                      use_colnames=True, max_len=4)
fpgrowth_co_time = time.time() - t0

co_rules_fp = association_rules(
    co_freq_fp, metric='confidence', min_threshold=MIN_CONF
)

print(f'Min support          : {MIN_SUP_CO}')
print(f'Min confidence       : {MIN_CONF}')
print(f'Frequent itemsets    : {len(co_freq_fp):,}')
print(f'Rules generated      : {len(co_rules_fp):,}')
print(f'Runtime              : {fpgrowth_co_time:.2f}s')
print()

top10_fp = co_rules_fp.nlargest(10, 'lift')[['antecedents','consequents','support','confidence','lift']]
top10_fp['antecedents'] = top10_fp['antecedents'].apply(lambda s: ', '.join(sorted(s)))
top10_fp['consequents'] = top10_fp['consequents'].apply(lambda s: ', '.join(sorted(s)))
print('Top 10 Co-Occurrence Rules by Lift (FP-Growth):')
print(top10_fp.round(4).to_string(index=False))

### 3-B  FP-Growth — Price Spike Transactions

In [ ]:
print('Running FP-Growth on Price Spike transactions ...')
t0 = time.time()
spike_freq_fp = fpgrowth(spike_enc, min_support=MIN_SUP_SPIKE,
                         use_colnames=True, max_len=4)
fpgrowth_spike_time = time.time() - t0

spike_rules_fp = association_rules(
    spike_freq_fp, metric='confidence', min_threshold=MIN_CONF
)

print(f'Min support          : {MIN_SUP_SPIKE}')
print(f'Min confidence       : {MIN_CONF}')
print(f'Frequent itemsets    : {len(spike_freq_fp):,}')
print(f'Rules generated      : {len(spike_rules_fp):,}')
print(f'Runtime              : {fpgrowth_spike_time:.2f}s')
print()

top10_fps = spike_rules_fp.nlargest(10, 'lift')[['antecedents','consequents','support','confidence','lift']]
top10_fps['antecedents'] = top10_fps['antecedents'].apply(lambda s: ', '.join(sorted(s)))
top10_fps['consequents'] = top10_fps['consequents'].apply(lambda s: ', '.join(sorted(s)))
print('Top 10 Price Spike Rules by Lift (FP-Growth):')
print(top10_fps.round(4).to_string(index=False))

## 4. Algorithm Comparison: Apriori vs FP-Growth

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Algorithm'       : ['Apriori','Apriori','FP-Growth','FP-Growth'],
    'Transaction Type': ['Co-Occurrence','Price Spike','Co-Occurrence','Price Spike'],
    'Input Rows'      : [len(co_samp), len(spike_samp), len(co_enc), len(spike_enc)],
    'Itemsets Found'  : [len(co_freq_ap), len(spike_freq_ap), len(co_freq_fp), len(spike_freq_fp)],
    'Rules Generated' : [len(co_rules_ap), len(spike_rules_ap), len(co_rules_fp), len(spike_rules_fp)],
    'Runtime (s)'     : [round(apriori_co_time,2), round(apriori_spike_time,2),
                         round(fpgrowth_co_time,2), round(fpgrowth_spike_time,2)],
})
print('=== Algorithm Comparison ===')
print(comparison.to_string(index=False))
print()
print(f'FP-Growth speedup (Co-Occurrence): {apriori_co_time/max(fpgrowth_co_time,0.001):.1f}x faster')
print(f'FP-Growth speedup (Price Spike)  : {apriori_spike_time/max(fpgrowth_spike_time,0.001):.1f}x faster')

# ── Side-by-side bar charts ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ALGOS  = ['Apriori\n(Co-Occ)', 'Apriori\n(Spike)', 'FP-Growth\n(Co-Occ)', 'FP-Growth\n(Spike)']
COLORS = ['#457B9D','#A8DADC','#E63946','#F4A261']

# Runtime
runtimes = [apriori_co_time, apriori_spike_time, fpgrowth_co_time, fpgrowth_spike_time]
axes[0].bar(ALGOS, runtimes, color=COLORS, edgecolor='white')
for i, v in enumerate(runtimes):
    axes[0].text(i, v + max(runtimes)*0.01, f'{v:.2f}s', ha='center', fontsize=9)
axes[0].set_title('Runtime Comparison', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Seconds')
axes[0].grid(axis='y', linestyle=':', alpha=0.5)

# Rules generated
n_rules = [len(co_rules_ap), len(spike_rules_ap), len(co_rules_fp), len(spike_rules_fp)]
axes[1].bar(ALGOS, n_rules, color=COLORS, edgecolor='white')
for i, v in enumerate(n_rules):
    axes[1].text(i, v + max(n_rules)*0.01, f'{v:,}', ha='center', fontsize=9)
axes[1].set_title('Rules Generated', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Rules')
axes[1].grid(axis='y', linestyle=':', alpha=0.5)

fig.suptitle('Apriori vs FP-Growth — Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('algo_comparison.png', dpi=150)
plt.show()

## 5. Visualizations

### 5-A  Support–Confidence–Lift Bubble Plot

Each bubble is one association rule. **Size** = lift, **colour** = lift intensity.
Stronger rules (higher lift) appear as larger, darker bubbles.

In [ ]:
def bubble_plot(rules, title, ax):
    sc = ax.scatter(
        rules['support'], rules['confidence'],
        s=rules['lift'] * 60,
        c=rules['lift'],
        cmap='RdYlGn', alpha=0.7, edgecolors='white', linewidth=0.5
    )
    ax.set_xlabel('Support', fontsize=10)
    ax.set_ylabel('Confidence', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(linestyle=':', alpha=0.4)
    plt.colorbar(sc, ax=ax, label='Lift')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bubble_plot(co_rules_fp,    'Co-Occurrence Rules (FP-Growth)', axes[0])
bubble_plot(spike_rules_fp, 'Price Spike Rules (FP-Growth)',   axes[1])

fig.suptitle('Support vs Confidence — Bubble Size & Colour = Lift',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bubble_plot.png', dpi=150)
plt.show()

### 5-B  Top 15 Rules by Lift

In [ ]:
def fmt(frozen_set):
    items = sorted(frozen_set)
    return ' + '.join(items) if len(items) <= 2 else items[0] + ' + ...' 

def top_rules_bar(rules, title, ax, color):
    top = rules.nlargest(15, 'lift').copy()
    top['rule'] = (top['antecedents'].apply(fmt) + '\n→  ' + top['consequents'].apply(fmt))
    top = top.sort_values('lift')
    bars = ax.barh(top['rule'], top['lift'], color=color, edgecolor='white')
    ax.axvline(1.0, color='grey', linestyle='--', alpha=0.6, label='Lift = 1 (random)')
    for bar, val in zip(bars, top['lift']):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Lift')
    ax.legend(fontsize=8)
    ax.grid(axis='x', linestyle=':', alpha=0.4)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
top_rules_bar(co_rules_fp,    'Top 15 Co-Occurrence Rules (FP-Growth)', axes[0], '#457B9D')
top_rules_bar(spike_rules_fp, 'Top 15 Price Spike Rules (FP-Growth)',   axes[1], '#E63946')
fig.suptitle('Top Association Rules by Lift', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('top_rules_bar.png', dpi=150)
plt.show()

### 5-C  Association Rule Network Graph

Nodes = items, directed edges = rules. **Arrow thickness** ∝ confidence. **Node size** ∝ total support.

In [ ]:
def draw_network(rules, title, top_n=25, min_lift=1.0):
    top = (
        rules[rules['lift'] >= min_lift]
        .nlargest(top_n, 'lift')
        .copy()
    )

    G = nx.DiGraph()
    node_support = {}

    for _, row in top.iterrows():
        ant = ', '.join(sorted(row['antecedents']))
        con = ', '.join(sorted(row['consequents']))
        G.add_edge(ant, con,
                   weight=row['confidence'],
                   lift=row['lift'])
        node_support[ant] = node_support.get(ant, 0) + row['support']
        node_support[con] = node_support.get(con, 0) + row['support']

    if len(G.nodes) == 0:
        print(f'No rules with lift >= {min_lift} found for network graph.')
        return

    pos = nx.spring_layout(G, seed=42, k=2.5)
    fig, ax = plt.subplots(figsize=(14, 9))

    node_sizes = [node_support.get(n, 0.01) * 6000 + 400 for n in G.nodes]
    edge_widths = [G[u][v]['weight'] * 5 for u, v in G.edges]
    edge_colors = [G[u][v]['lift'] for u, v in G.edges]

    nx.draw_networkx_nodes(G, pos, node_size=node_sizes,
                           node_color='#457B9D', alpha=0.85, ax=ax)
    edges = nx.draw_networkx_edges(
        G, pos,
        width=edge_widths,
        edge_color=edge_colors,
        edge_cmap=plt.cm.YlOrRd,
        arrowsize=20,
        connectionstyle='arc3,rad=0.1',
        ax=ax
    )
    nx.draw_networkx_labels(G, pos, font_size=7, font_color='white',
                            font_weight='bold', ax=ax)

    sm = plt.cm.ScalarMappable(cmap=plt.cm.YlOrRd,
                                norm=plt.Normalize(min(edge_colors), max(edge_colors)))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Lift', shrink=0.8)

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    fname = title.lower().replace(' ', '_')[:30] + '_network.png'
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f'  Nodes: {len(G.nodes)}  |  Edges: {len(G.edges)}')

draw_network(co_rules_fp,    'Co-Occurrence Rule Network',  top_n=30)
draw_network(spike_rules_fp, 'Price Spike Rule Network',    top_n=30)

### 5-D  Regional Price Spike Heatmap

How often does each commodity spike in each region? Darker cells = more frequent spikes.

In [ ]:
# Count spike occurrences per (region, item)
spike_counts = (
    df_sp[df_sp['is_spike']]
    .groupby(['adm1_name', 'item'])
    .size()
    .reset_index(name='spike_count')
)

# Top 20 most frequently spiking items
top_items = (
    spike_counts.groupby('item')['spike_count'].sum()
    .nlargest(20).index.tolist()
)

heat_data = (
    spike_counts[spike_counts['item'].isin(top_items)]
    .pivot_table(index='adm1_name', columns='item',
                 values='spike_count', fill_value=0)
)

# Shorten region names for display
heat_data.index = (heat_data.index
                   .str.replace('Autonomous region in Muslim Mindanao', 'ARMM')
                   .str.replace('Cordillera Administrative region', 'CAR')
                   .str.replace('National Capital region', 'NCR')
                   .str.replace(r'Region (\w+) \((.+?)\)', lambda m: m.group(1), regex=True))

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(
    heat_data,
    cmap='YlOrRd',
    linewidths=0.3,
    annot=False,
    fmt='d',
    ax=ax,
    cbar_kws={'label': 'Spike Count'}
)
ax.set_title('Regional Price Spike Frequency — Top 20 Commodities',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Commodity', fontsize=10)
ax.set_ylabel('Region', fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('spike_heatmap.png', dpi=150)
plt.show()

## 6. Summary

In [ ]:
print('=' * 65)
print('  CHAPTER 9 PIPELINE SUMMARY')
print('=' * 65)
print()
print('TRANSACTION ENGINEERING')
print(f'  Co-Occurrence baskets  : {len(co_transactions):,}  (market x date)')
print(f'  Price Spike baskets    : {len(spike_transactions):,}  (region x date, p75 threshold)')
print()
print('ALGORITHM COMPARISON')
print(f'  {"":<22} {"Apriori":>12} {"FP-Growth":>12}')
print(f'  {"Co-Occ Itemsets":<22} {len(co_freq_ap):>12,} {len(co_freq_fp):>12,}')
print(f'  {"Co-Occ Rules":<22} {len(co_rules_ap):>12,} {len(co_rules_fp):>12,}')
print(f'  {"Co-Occ Runtime (s)":<22} {apriori_co_time:>12.2f} {fpgrowth_co_time:>12.2f}')
print(f'  {"Spike Itemsets":<22} {len(spike_freq_ap):>12,} {len(spike_freq_fp):>12,}')
print(f'  {"Spike Rules":<22} {len(spike_rules_ap):>12,} {len(spike_rules_fp):>12,}')
print(f'  {"Spike Runtime (s)":<22} {apriori_spike_time:>12.2f} {fpgrowth_spike_time:>12.2f}')
print()
print('KEY INSIGHTS')
print('  - FP-Growth is faster than Apriori while producing the same rule set')
print('  - Co-Occurrence rules reveal which commodities share market presence')
print('  - Price Spike rules reveal commodities whose prices move together')
print('  - High-lift spike rules can warn of inflationary clusters across')
print('    food categories, useful for government intervention planning')
print()
print('OUTPUTS SAVED')
for f in ['algo_comparison.png','bubble_plot.png','top_rules_bar.png',
          'co-occurrence_rule_network.png','price_spike_rule_network.png',
          'spike_heatmap.png']:
    print(f'  {f}')